# Reconhecimento de entidades nomeadas

In [ ]:
#!pip install -U spacy
#!python -m spacy download pt_core_news_sm
#!python -m spacy download en_core_web_sm

In [ ]:
import zipfile
import spacy
import pt_core_news_sm
import en_core_web_sm
import pandas as pd
import random
from tqdm import tqdm
from spacy.training import Example

In [ ]:
with zipfile.ZipFile('/content/texts.zip', 'r') as f:
  print(*f.namelist(), sep='\n')

ADI2TJDFT.txt
adi3767.txt
Ag10000170733596001.txt
Ag10105170208398001.txt
AgAIRR11889820145030011.txt
AgCr10582160008758001.txt
AgRgSTJ1.txt
AgRgSTJ2.txt
AgRgTSE1.txt
AgRgTSE3.txt
AIAgRAgI6193ARAGUARIMG.txt
airr801422012.txt
AIRR3999520145020086.txt
AIRR15708820115050222.txt
AP00000794920137060006.txt
AP00001415620157010201.txt
AP00001441420167030203.txt
AP771420167080008PA.txt
CP32320177080008PA.txt
DespSEPLAGDF.txt
ED1STM.txt
ED1TJAC.txt
EDAgRgTSE2.txt
EDEDARR208420135040232.txt
EDRR1TST.txt
EEDRR9715120105020002.txt
ERR731004520105130003.txt
HC110260SP.txt
HC151914AgRES.txt
HC340624SP.txt
HC418951PR.txt
HC70000845920187000000.txt
lei11340.txt
Lei11788.txt
LoaDF2018.txt
Pet128TSE5.txt
Port77DF.txt
Rcl3495STJ.txt
REE5908TSE4.txt
REsp1583083RS.txt
RR474820145230056.txt
RR942006420095040028.txt
RR2574407120025020372.txt
TCU4687.txt
TSTRR16037920105200001.txt
AC1TCU.txt
AC1TJAC.txt
AC1TJMG.txt
AC2.txt
ACORDAOTCU25052016.txt


In [ ]:
with zipfile.ZipFile('/content/texts.zip', 'r') as f:
  with f.open('ADI2TJDFT.txt') as arq:
    texto = arq.read().decode('utf-8')

print(texto)

Órgão	:	Conselho Especial
Classe	:	ADI  Ação Direta de Inconstitucionalidade
N. Processo	:	2010002019357-4
Requerente(s)	:	PROCURADORA-GERAL DE JUSTIÇA DO DISTRITO FEDERAL E TERRITÓRIOS
Requerido(s)	:	PRESIDENTE DA CÂMARA LEGISLATIVA DO DISTRITO FEDERAL E OUTRO(S)
Relator 	:	Desembargador LÉCIO RESENDE
	EMENTA	
AÇÃO DIRETA DE INCONSTITUCIONALIDADE. LEIS DISTRITAIS N.º 747/1994 E 2018/1998. LEI COMPLEMENTAR DISTRITAL N.º 380/2001. INCONSTITUCIONALIDADE FORMAL. LEI ORGÂNICA DO DISTRITO FEDERAL. OCUPAÇÃO DE ÁREA PÚBLICA. COMPETÊNCIA PRIVATIVA DO GOVERNADOR DO DISTRITO FEDERAL. AÇÃO JULGADA PROCEDENTE EM RAZÃO DO VÍCIO FORMAL. Tanto o Decreto n.º 10.829/87, quanto a Portaria n.º 314/92, do Instituto Brasileiro do Patrimônio Cultural  IBPC, hoje Instituto do Patrimônio Histórico e Artístico Nacional  IPHAN, conferem ao Governador do Distrito Federal competência privativa para iniciar o processo legislativo, quando se tratar o tema de uso e ocupação do solo em todo o território do Distrito F

In [ ]:
model_ner = pt_core_news_sm.load()
doc = model_ner(texto)


entidades = []
labels = []

for ent in doc.ents:
  entidades.append(ent.text)
  labels.append(ent.label_)

pd.DataFrame({'entidade': entidades, 'label': labels})

,entidade,label
0,Órgão,LOC
1,Conselho Especial\nClasse,ORG
2,DISTRITO FEDERAL E TERRITÓRIOS\nRequerido(s,LOC
3,DA CÂMARA,ORG
4,DISTRITO FEDERAL E OUTRO(S)\nRelator,LOC
...,...,...
851,PROCEDENTE,LOC
852,Leis Distritais,MISC
853,Lei Complementar,MISC
854,omnes,ORG


In [ ]:
model_ner.get_pipe('ner').labels

('LOC', 'MISC', 'ORG', 'PER')

In [ ]:
print('LOC', spacy.explain('LOC'))
print('PER', spacy.explain('PER'))
print('ORG', spacy.explain('ORG'))
print('MISC', spacy.explain('MISC'))

LOC Non-GPE locations, mountain ranges, bodies of water
PER Named person or family.
ORG Companies, agencies, institutions, etc.
MISC Miscellaneous entities, e.g. events, nationalities, products or works of art


In [ ]:
spacy.displacy.render(doc, style='ent', jupyter=True)

In [ ]:
import en_core_web_sm

model_ner_ing = spacy.load('en_core_web_sm')
model_ner_ing.get_pipe('ner').labels

('CARDINAL',
 'DATE',
 'EVENT',
 'FAC',
 'GPE',
 'LANGUAGE',
 'LAW',
 'LOC',
 'MONEY',
 'NORP',
 'ORDINAL',
 'ORG',
 'PERCENT',
 'PERSON',
 'PRODUCT',
 'QUANTITY',
 'TIME',
 'WORK_OF_ART')

In [ ]:
dados = []

with zipfile.ZipFile('/content/texts.zip') as f:
  for nome_arq in f.namelist():
    with f.open(nome_arq) as arquivo:
      conteudo = arquivo.read().decode('utf-8')
      palavras = conteudo.split()
      for palavra in palavras:
        dados.append([nome_arq, palavra])

tabela_palavras = pd.DataFrame(dados, columns = ['arquivo', 'palavra'])
tabela_palavras

,arquivo,palavra
0,ADI2TJDFT.txt,Órgão
1,ADI2TJDFT.txt,:
2,ADI2TJDFT.txt,Conselho
3,ADI2TJDFT.txt,Especial
4,ADI2TJDFT.txt,Classe
...,...,...
193487,ACORDAOTCU25052016.txt,Ministro-Substituto
193488,ACORDAOTCU25052016.txt,AUGUSTO
193489,ACORDAOTCU25052016.txt,SHERMAN
193490,ACORDAOTCU25052016.txt,CAVALCANTI


In [ ]:
tabela_palavras.to_csv('palavras.csv', index=False, sep = '\t')

### Rotulação

O formato IOB (Inside-Outside-Beginning) é amplamente utilizado para rotular sequências de texto em tarefas de NLP, principalmente em tarefas de reconhecimento de entidades nomeadas (NER). Este formato ajuda a identificar palavras ou frases que pertencem a categorias específicas, como nomes de pessoas, organizações, locais e datas.

Esse formato é um esquema de rotulação que classifica as palavras de uma sequência de texto em três categorias:

- B- (Beginning): indica o início de uma entidade. É usado para marcar a primeira palavra de uma entidade nomeada.
- I- (Inside): indica que a palavra faz parte de uma entidade, mas não é a primeira palavra. Todas as palavras subsequentes que ainda pertencem à mesma entidade são marcadas com "I".
- O (Outside): indica que a palavra não faz parte de nenhuma entidade nomeada. É utilizada para palavras que não pertencem a nenhuma categoria específica de interesse.

**Vantagens do formato IOB**

Simplicidade: é fácil de entender e aplicar. Cada palavra recebe um rótulo que indica claramente se pertence a uma entidade e, em caso positivo, se é o início ou parte de uma entidade.
- Versatilidade: pode ser utilizado para diferentes tipos de entidades, bastando adaptar os rótulos (por exemplo, B-PESSOA, I-PESSOA, B-LOCAL, I-LOCAL, B-CARGO, I-CARGO).
- Facilidade de treinamento: é um formato muito utilizado em modelos de aprendizado de máquina para NLP, pois fornece informações ricas e estruturadas sobre as entidades no texto.

**Limitações do formato IOB**

Embora seja amplamente utilizado, o formato IOB tem algumas limitações:
- Confusão em casos ambíguos: em algumas situações, pode ser difícil decidir onde começa e termina uma entidade, especialmente em textos complexos.
- Depende de segmentação prévia: para utilizar o formato IOB, é necessário que o texto esteja devidamente tokenizado (dividido em palavras ou subpalavras).

**Alternativas ao formato IOB**

Existem variações do formato IOB, como o IOB2 e o BIOES:

- IOB2: semelhante ao IOB, mas todos os rótulos de entidade começam com "B-" independentemente de serem a primeira palavra de uma entidade.
- BIOES: adiciona dois rótulos adicionais: "E-" (End), que indica a última palavra de uma entidade, e "S-" (Single), que marca uma entidade que consiste em uma única palavra.

In [ ]:
tabela_palavras = pd.read_csv('/content/palavras_IOB.tsv', sep='\t')
tabela_palavras

,arquivo,palavra,label
0,TCU4687.txt,GRUPO,O
1,TCU4687.txt,I,O
2,TCU4687.txt,CLASSE,O
3,TCU4687.txt,II,O
4,TCU4687.txt,2ª,B-ORGANIZACAO
...,...,...,...
229272,RR2574407120025020372.txt,de,O
229273,RR2574407120025020372.txt,Chaves,O
229274,RR2574407120025020372.txt,Públicas,O
229275,RR2574407120025020372.txt,Brasileira,O


In [ ]:
tabela_palavras['label'].unique()

array(['O', 'B-ORGANIZACAO', 'I-ORGANIZACAO', 'B-JURISPRUDENCIA',
       'I-JURISPRUDENCIA', 'B-PESSOA', 'I-PESSOA', 'B-LEGISLACAO',
       'I-LEGISLACAO', 'B-TEMPO', 'B-LOCAL', 'I-LOCAL', 'I-TEMPO'],
      dtype=object)

In [ ]:
# converter para formato spacy

grupos_arquivos = tabela_palavras.groupby('arquivo')
grupos_arquivos.get_group('ADI2TJDFT.txt')

,arquivo,palavra,label
46105,ADI2TJDFT.txt,Órgão,O
46106,ADI2TJDFT.txt,:,O
46107,ADI2TJDFT.txt,Conselho,B-ORGANIZACAO
46108,ADI2TJDFT.txt,Especial,I-ORGANIZACAO
46109,ADI2TJDFT.txt,Classe,O
...,...,...,...
55911,ADI2TJDFT.txt,a,O
55912,ADI2TJDFT.txt,ação,O
55913,ADI2TJDFT.txt,",",O
55914,ADI2TJDFT.txt,maioria,O


In [ ]:
tabela_agrupada = grupos_arquivos.get_group('ADI2TJDFT.txt')[['palavra', 'label']].values
conteudo = ''
anotacoes = {'entities': []}
inicio_palavra = 0
fim_palavra = 0
for texto, label in tabela_agrupada:
    texto = str(texto)
    tamanho_texto = len(texto)+1

    inicio_palavra = fim_palavra
    fim_palavra = inicio_palavra + tamanho_texto

    if label != 'O':
        anotacao = (inicio_palavra, fim_palavra-1, label)
        anotacoes['entities'].append(anotacao)

    conteudo = conteudo + texto + ' '

In [ ]:
anotacoes

{'entities': [(8, 16, 'B-ORGANIZACAO'),
  (17, 25, 'I-ORGANIZACAO'),
  (91, 106, 'B-JURISPRUDENCIA'),
  (158, 166, 'B-LOCAL'),
  (167, 174, 'I-LOCAL'),
  (221, 227, 'B-ORGANIZACAO'),
  (228, 239, 'I-ORGANIZACAO'),
  (240, 242, 'I-ORGANIZACAO'),
  (243, 251, 'I-ORGANIZACAO'),
  (252, 259, 'I-ORGANIZACAO'),
  (298, 303, 'B-PESSOA'),
  (304, 311, 'I-PESSOA'),
  (358, 362, 'B-LEGISLACAO'),
  (363, 373, 'I-LEGISLACAO'),
  (374, 377, 'I-LEGISLACAO'),
  (378, 386, 'I-LEGISLACAO'),
  (389, 398, 'B-LEGISLACAO'),
  (401, 404, 'B-LEGISLACAO'),
  (405, 417, 'I-LEGISLACAO'),
  (418, 427, 'I-LEGISLACAO'),
  (428, 431, 'I-LEGISLACAO'),
  (432, 440, 'I-LEGISLACAO'),
  (474, 477, 'B-LEGISLACAO'),
  (478, 486, 'I-LEGISLACAO'),
  (487, 489, 'I-LEGISLACAO'),
  (490, 498, 'I-LEGISLACAO'),
  (499, 506, 'I-LEGISLACAO'),
  (575, 583, 'B-LOCAL'),
  (584, 591, 'I-LOCAL'),
  (653, 660, 'B-LEGISLACAO'),
  (661, 664, 'I-LEGISLACAO'),
  (665, 674, 'I-LEGISLACAO'),
  (686, 694, 'B-LEGISLACAO'),
  (695, 698, 'I-LEGIS

In [ ]:
conteudo.find('Conselho Especial')

8

In [ ]:
conteudo.find('Conselho Especial') + len('Conselho Especial')

25

### Transformando todos os dados para formato correto

In [ ]:
arquivos = grupos_arquivos.groups.keys()
arquivos

dict_keys(['AC1TCU.txt', 'AC1TJAC.txt', 'AC1TJMG.txt', 'AC2.txt', 'ACORDAOTCU25052016.txt', 'ADI2TJDFT.txt', 'AIAgRAgI6193ARAGUARIMG.txt', 'AIRR15708820115050222.txt', 'AIRR3999520145020086.txt', 'AP00000794920137060006.txt', 'AP00001415620157010201.txt', 'AP00001441420167030203.txt', 'AP771420167080008PA.txt', 'Ag10000170733596001.txt', 'Ag10105170208398001.txt', 'AgAIRR11889820145030011.txt', 'AgCr10582160008758001.txt', 'AgRgSTJ1.txt', 'AgRgSTJ2.txt', 'AgRgTSE1.txt', 'AgRgTSE3.txt', 'CP32320177080008PA.txt', 'DespSEPLAGDF.txt', 'ED1STM.txt', 'ED1TJAC.txt', 'EDAgRgTSE2.txt', 'EDEDARR208420135040232.txt', 'EDRR1TST.txt', 'EEDRR9715120105020002.txt', 'ERR731004520105130003.txt', 'HC110260SP.txt', 'HC151914AgRES.txt', 'HC340624SP.txt', 'HC418951PR.txt', 'HC70000845920187000000.txt', 'Lei11788.txt', 'LoaDF2018.txt', 'Pet128TSE5.txt', 'Port77DF.txt', 'REE5908TSE4.txt', 'REsp1583083RS.txt', 'RR2574407120025020372.txt', 'RR474820145230056.txt', 'RR942006420095040028.txt', 'Rcl3495STJ.txt', 

In [ ]:
documentos = []
for arquivo in arquivos:
    documento = []
    tabela_agrupada = grupos_arquivos.get_group(arquivo)[['palavra', 'label']].values
    conteudo = ''
    anotacoes = {'entities': []}
    inicio_palavra = 0
    fim_palavra = 0
    for texto, label in tabela_agrupada:
        texto = str(texto)
        tamanho_texto = len(texto)+1

        inicio_palavra = fim_palavra
        fim_palavra = inicio_palavra + tamanho_texto

        if label != 'O':
            anotacao = (inicio_palavra, fim_palavra-1, label)
            anotacoes['entities'].append(anotacao)

        conteudo = conteudo + texto + ' '

    documento = (conteudo, anotacoes)
    documentos.append(documento)

In [ ]:
documentos

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
random.shuffle(documentos)

In [ ]:
print(type(documentos[2]))

<class 'tuple'>


In [ ]:
dados_train = documentos[:40]
dados_validacao = documentos[40:]

In [ ]:
dados_train

('Brastra.gif ( 4376 bytes ) Presidência da República Casa Civil Subchefia para Assuntos Jurídicos LEI Nº 11.340 , DE 7 DE AGOSTO DE 2006 . Vigência ( Vide ADI nº 4427 ) Cria mecanismos para coibir a violência doméstica e familiar contra a mulher , nos termos do § 8o do art . 226 da Constituição Federal , da Convenção sobre a Eliminação de Todas as Formas de Discriminação contra as Mulheres e da Convenção Interamericana para Prevenir , Punir e Erradicar a Violência contra a Mulher ; dispõe sobre a criação dos Juizados de Violência Doméstica e Familiar contra a Mulher ; altera o Código de Processo Penal , o Código Penal e a Lei de Execução Penal ; e dá outras providências . O PRESIDENTE DA REPÚBLICA Faço saber que o Congresso Nacional decreta e eu sanciono a seguinte Lei : TÍTULO I DISPOSIÇÕES PRELIMINARES Art . 1o Esta Lei cria mecanismos para coibir e prevenir a violência doméstica e familiar contra a mulher , nos termos do § 8o do art . 226 da Constituição Federal , da Convenção sobr

In [ ]:
def train_model_ner(dados_train, dados_validacao, epochs):
  model = spacy.load('pt_core_news_sm')
  if 'ner' not in model.pipe_names:
    ner = model.create_pipe('ner')
    model.add_pipe(ner, last=True)
  else:
    ner = model.get_pipe('ner')

  for _, anotacoes in dados_train:
    for ent in anotacoes.get('entities'):
      ner.add_label(ent[2])

  other_pipes = [pipe for pipe in model.pipe_names if pipe != 'ner']

  with model.disable_pipes(*other_pipes):
    spacy.util.fix_random_seed()
    optimizer = model.create_optimizer()

    for epoch in tqdm(range(epochs), desc = 'Treinando modelo'):
      random.seed(44)
      random.shuffle(dados_train)
      losses = {'ner': 0.0}

      for texts, anotacoes in dados_train:
        example = Example.from_dict(model.make_doc(texts), anotacoes)
        model.update([example], drop = 0.2, sgd=optimizer, losses=losses)

      print(f'\nEPOCH: {epoch+1} - LOSS médio de treino: {losses["ner"]/len(dados_train)}')

      val_losses = {'ner': 0.0}
      examples = []
      for texts, anotacoes in dados_validacao:
        example = Example.from_dict(model.make_doc(texts), anotacoes)
        examples.append(example)

      for example in examples:
        model.update(examples, drop=0, sgd=None, losses=val_losses)

      print(f'\nEPOCH: {epoch+1} - LOSS médio de treino: {val_losses["ner"]/len(dados_validacao)}')

  return model


Utilização de modelos pré-treinados:

1. Economia de tempo: treinar um modelo do zero exige muitos dados e tempo de processamento, enquanto os modelos pré-treinados já têm uma base pronta para uso.
2. Maior precisão: os modelos pré-treinados são ajustados com grandes volumes de dados, resultando em uma compreensão melhor das nuances do idioma.
3. Fácil customização: embora os modelos pré-treinados possam ser usados como estão, é possível ajustá-los para tarefas específicas (como NER em documentos jurídicos), adicionando novas entidades ou ajustando os parâmetros.

## Treinando e salvando modelo

In [ ]:
model_ner = train_model_ner(dados_train, dados_validacao, epochs = 30)

Treinando modelo:   0%|          | 0/30 [00:00<?, ?it/s]


EPOCH: 1 - LOSS médio de treino: 1001.248046875


Treinando modelo:   3%|▎         | 1/30 [01:43<50:07, 103.72s/it]


EPOCH: 1 - LOSS médio de treino: 7366.34912109375

EPOCH: 2 - LOSS médio de treino: 598.9879760742188


Treinando modelo:   7%|▋         | 2/30 [03:24<47:38, 102.08s/it]

...


EPOCH: 28 - LOSS médio de treino: 77.51719665527344

EPOCH: 29 - LOSS médio de treino: 46.893821716308594


Treinando modelo:  97%|█████████▋| 29/30 [50:32<01:41, 101.22s/it]


EPOCH: 29 - LOSS médio de treino: 67.18702697753906

EPOCH: 30 - LOSS médio de treino: 45.55202102661133


Treinando modelo: 100%|██████████| 30/30 [52:12<00:00, 104.41s/it]


EPOCH: 30 - LOSS médio de treino: 74.9092788696289


In [ ]:
model_ner.to_disk('model_ner')

### Salvando e carregando modelo

In [ ]:
model_ner = spacy.load('/content/model_ner')

In [ ]:
!zip -r model_ner.zip '/content/model_ner'

  adding: content/model_ner/ (stored 0%)
  adding: content/model_ner/tokenizer (deflated 84%)
  adding: content/model_ner/senter/ (stored 0%)
  adding: content/model_ner/senter/cfg (stored 0%)
  adding: content/model_ner/senter/model (deflated 10%)
  adding: content/model_ner/lemmatizer/ (stored 0%)
  adding: content/model_ner/lemmatizer/cfg (deflated 76%)
  adding: content/model_ner/lemmatizer/trees (deflated 80%)
  adding: content/model_ner/lemmatizer/model (deflated 7%)
  adding: content/model_ner/morphologizer/ (stored 0%)
  adding: content/model_ner/morphologizer/cfg (deflated 92%)
  adding: content/model_ner/morphologizer/model (deflated 7%)
  adding: content/model_ner/config.cfg (deflated 74%)
  adding: content/model_ner/meta.json (deflated 87%)
  adding: content/model_ner/ner/ (stored 0%)
  adding: content/model_ner/ner/cfg (deflated 33%)
  adding: content/model_ner/ner/moves (deflated 80%)
  adding: content/model_ner/ner/model (deflated 7%)
  adding: content/model_ner/parser/ 

In [ ]:
import requests

url = 'https://cdn3.gnarususercontent.com.br/3972-nlp/Projeto/dados/20150110436469APC.txt'
resposta = requests.get(url)

with open('20150110436469APC.txt', 'w', encoding='utf-8') as f:
    f.write(resposta.text)

In [ ]:
with open('/content/20150110436469APC.txt', encoding='utf-8') as f:
  text = f.read()

In [ ]:
print(text)

E M E N T A
Poder JudiciÃ¡rio da UniÃ£o
TRIBUNAL DE JUSTIÃA DO DISTRITO FEDERAL E TERRITÃRIOS Fls. _____
ÃrgÃ£o : 8Âª TURMA CÃVEL
Classe : APELAÃÃO CÃVEL
N. Processo : 20150110436469APC
(0012843-03.2015.8.07.0001)
Apelante(s) : BRASILIA CURSOS E CONCURSOS LTDA
GRANCURSOS ESCOLA PARA
CONCURSOS PUBLICOS LTDA
Apelado(s) : ALISSON SILVA BATISTA DE MORAES
Relatora : Desembargadora NÃDIA CORRÃA LIMA
AcÃ³rdÃ£o N. : 1082726
CIVIL E PROCESSUAL CIVIL. AÃÃO MONITÃRIA. CITAÃÃO
REALIZADA APÃS O DECURSO DO PRAZO
PRESCRICIONAL. PRESCRIÃÃO DA PRETENSÃO
MONITÃRIA. RECONHECIMENTO. MANUTENÃÃO DA
SENTENÃA.
1. Nos termos da SÃºmula nÂº 503 do colendo Superior Tribunal
de JustiÃ§a, "O prazo para ajuizamento de aÃ§Ã£o monitÃ³ria em
face do emitente de cheque sem forÃ§a executiva Ã© quinquenal,
a contar do dia seguinte Ã  data de emissÃ£o estampada na
cÃ¡rtula".
2. Evidenciado que a citaÃ§Ã£o somente foi aperfeiÃ§oada apÃ³s o
decurso do prazo prescricional, tem-se por correta a extinÃ§Ã£

In [ ]:
!unzip model_ner.zip -d '/content/model_ner'

In [ ]:
doc = model_ner(text)

spacy.displacy.render(doc, style = 'ent', jupyter = True)

In [ ]:
rotulos = list(model_ner.get_pipe('ner').labels)
rotulos

In [ ]:
cores = {
 'B-JURISPRUDENCIA': '#F0F8FF',
 'B-LEGISLACAO': '#FA8072',
 'B-LOCAL': '#98FB98',
 'B-ORGANIZACAO': '#DDA0DD',
 'B-PESSOA': '#F0E68C',
 'B-TEMPO': '#FFB6C1',
 'I-JURISPRUDENCIA': '#F0F8FF',
 'I-LEGISLACAO': '#FA8072',
 'I-LOCAL': '#98FB98',
 'I-ORGANIZACAO': '#DDA0DD',
 'I-PESSOA': '#F0E68C',
 'I-TEMPO': '#FFB6C1',
 'LOC': '#D3D3D3',
 'MISC': '#D3D3D3',
 'ORG': '#D3D3D3',
 'PER': '#D3D3D3'
}
opcoes = {'ents': rotulos, 'colors': cores}

spacy.displacy.render(doc, style = 'ent', jupyter = True, options = opcoes)

## Pontos importantes


- Realizar a leitura de arquivos de texto a partir de um arquivo compactado;
- Entender o conceito do reconhecimento de entidades nomeadas e exemplos de aplicações;
- Fazer o download e aplicar um modelo NER da biblioteca SpaCy;
- Listar e tabelar todas as entidades nomeadas de um documento usando oSpaCy;
- Destacar entidades nomeadas em um texto a partir de um método de renderização do Displacy;
- Realizar a tokenização de textos e armazenar o resultado em uma tabela;
- Rotular dados para o reconhecimento de entidades nomeadas no formato IOB;
- Transformar os dados para o formato adequado, com o texto dos documentos e posição das entidades no texto;
- Realizar o processo de divisão dos dados de treinamento e validação;
- Configurar o pipeline de um modelo NER em uma função de treinamento;
- Criar o loop de treinamento e validação para um modelo de NLP;
- Utilizar modelos pré-treinados e aprimorar sua aplicação com novos dados.
Realizar o treinamento de um modelo de NLP;
- Armazenar o modelo em um arquivo compactado para ser utilizado fora do ambiente do Google Collaboratory;
- Carregar o modelo treinado usando a biblioteca SpaCy;
- Testar o funcionamento do modelo em um novo documento para identificar entidades nomeadas;
- Alterar as cores da renderização do DisplaCy para destacar as entidades nomeadas reconhecidas;
- Preparar o ambiente virtual com as bibliotecas necessárias para utilização do Streamlit;
- Criar uma aplicação interativa utilizando o Streamlit;
- Realizar o reconhecimento de entidades nomeadas de forma simples a partir do upload de arquivos de texto;
- Utilizar a renderização do Spacy no ambiente do Streamlit para destacar entidades nomeadas diretamente no texto;